# Credit Card Fraud Detection — Proof of Concept

**Audience:** Senior leadership and the Fraud & Risk Committee. No technical background is assumed.<br>
**Prepared by:** Uday Kakkar<br>
**Source repository:** [https://github.com/udaykakkar/azureml-example](https://github.com/udaykakkar/azureml-example)<br>
**Status:** Proof of concept. **Not approved for production.** No live transaction is affected by anything in this document.

---

## What this document is

In our previous submission we recommended that the business address its fraud-detection
problem with machine learning. This document is the first working prototype built against
that recommendation, and it is the evidence on which you are being asked to make a funding
decision.

What you are reading is a **notebook**: a single document that holds three things side by
side — the working code, a plain-English explanation of what that code does, and the
results it produces. **You do not need to read any code.** Every grey code block below is
preceded by an explanation of what it does and why it matters commercially. If you read
only the explanations and skip every code block, you will still have the complete picture.

## The question you are being asked

> **Is this approach promising enough to fund a production build — and if so, what has to change first?**

## Bottom line up front

| Question | Short answer |
|---|---|
| **Does it work?** | Partly. It correctly identifies roughly **1 in 4** fraudulent transactions (28%). |
| **Is it accurate?** | It is "99.8% accurate" — and that number is **meaningless here**. See *The accuracy trap* below. |
| **Does it pay for itself today?** | Marginally. Net benefit of about **€13,583** across 284,807 transactions — roughly **€477k a year** at a portfolio of 10M transactions. |
| **Is that good enough?** | No. It captures only **18%** of the value available. A production-grade model should reach roughly **68%**. |
| **What is the recommendation?** | **Fund a pilot, not a production rollout.** The prototype proves the plumbing works; it does not yet prove the model is good enough to make decisions about real customers. |

## What we built, in one paragraph

We took a public file of 284,807 real card transactions, of which 492 are confirmed
frauds (0.173% of the total). We then taught a computer to spot transactions that look
**unusual** compared with everything else it has seen. Note carefully what that sentence
does *not* say: we did not teach it what fraud looks like. We only taught it what *normal*
looks like, and asked it to point at anything that stands out. Finally, we compared its
list of flagged transactions against the 492 frauds we already knew about, to see how
well "unusual" and "fraudulent" actually line up.

> **The single most important idea in this document:** what we have built is an
> **unusualness detector, not a fraud detector**. Most fraud is unusual, but most unusual
> things are not fraud. Every strength and every weakness described below follows from
> that one distinction.

## The accuracy trap

Please read this before you take away any other number.

Only about 1 transaction in 600 in this dataset is fraudulent. That means a system which
simply approves *everything*, catches no fraud whatsoever, and could be built in an
afternoon by an intern would be **99.8% accurate**. Our sophisticated model is
99.8% accurate. On the headline measure, doing nothing and doing this are
indistinguishable.

**So we never judge a fraud system on accuracy.** Two numbers matter instead, and they
should always be quoted together:

| Measure | Plain-English question it answers | Our result |
|---|---|---|
| **Catch rate** (recall) | Of all the fraud that really happened, how much did we find? | **28%** — we stop 138 frauds and let 354 through |
| **Hit rate** (precision) | Of everything we flagged, how much was really fraud? | **29%** — about 7 in 10 of our alarms are false |

Neither number is good. Both are a realistic starting point for a first prototype, and
both are fixable. The rest of this document explains how.

## How the system works, end to end

The diagram below shows the whole process. Everything inside the dashed blue boundary
happens inside a single secure, access-controlled, fully logged Azure environment — which
matters a great deal to our auditors, and is covered in the component table further down.

![Fraud detection pipeline — seven stages inside the Azure Machine Learning workspace](images/fraud_pipeline.png)

### The same story as a funnel, in real numbers

If you prefer numbers to boxes, this is what actually happened when we ran it:

```text
   284,807  transactions went in
      │
      │   the model flags anything that looks unusual
      ▼
       476  flagged for review  ──►  the queue your analysts would work
      │
      ├──►  138  really were fraud      TRUE HITS      value created
      └──►  338  were perfectly good    FALSE ALARMS   cost + annoyed customers

       354  frauds never flagged        MISSES         the expensive failure
```

Three things to take from that funnel:

1. **The queue is small and manageable.** 476 reviews out of 284,807 transactions is an
   entirely affordable workload — this is not a system that would swamp the operations team.
2. **The queue is mostly noise.** 338 of the 476 flagged customers did nothing wrong.
3. **The real problem is on the last line.** 354 frauds sailed straight through without
   ever being looked at. That, not the false alarms, is where the money is being lost —
   and the cost analysis after Step 5 puts a figure on it.

## What each part of Azure actually does

The word "Azure" covers a lot of separate services, and the code below touches several of
them. Rather than leave them as jargon, here is each component and the job it does. The
useful mental model is a **rented, high-security laboratory**: we do not own the building,
we do not buy the equipment, and we pay only for the hours we are actually inside.

| Azure ML component | Its role in this process |
|---|---|
| **Subscription** | The billing and governance boundary — the corporate account everything is charged to and controlled under. Think of it as the lease on the building. |
| **Workspace** | The single secure project room where the data, the code, the compute and the finished models all live together. Everything in the diagram above happens inside one workspace. |
| **`config.json`** | The key to that room. A small credentials file that tells our code which workspace to connect to and proves we are entitled to enter. It is deliberately kept out of the public code repository. |
| **Data asset** (`creditcard_fraud`) | The single governed, versioned copy of the transaction data. Nobody emails spreadsheets around; everyone points at this one registered copy, so we can always answer "exactly which data produced this result?" |
| **Datastore** | The underlying secure storage the data asset physically sits in, with encryption at rest and no public access. |
| **Compute instance / cluster** | The rented processing power that actually runs the work. It can be switched off when idle, which is the main lever on cost. |
| **Notebooks** (Authoring) | The workbench — the web-based editor where this very document is written and run. |
| **`azureml-core` SDK** | The remote control. The Python library that lets our code drive all of the above rather than clicking through a web interface by hand, which is what makes the process repeatable. |
| **Environment** (`conda.yaml`) | The recorded recipe of software versions, so the same run produces the same answer next month and next year. |
| **Model registry** | The vault of approved models. Each trained model is stored with a version number and a full history, so we can prove which version made which decision, and roll back instantly if one misbehaves. |
| **Managed endpoint** *(not used yet)* | How a registered model would eventually be exposed to live systems to score real transactions. Deliberately out of scope for this prototype. |
| **Azure Monitor / telemetry** | The automatic logging and audit trail. It is the source of the harmless status messages you may see printed under the code cells. |
| **Microsoft Entra ID + RBAC** | Identity and permissions — who is allowed to see the data, run the compute, or publish a model. This is what makes the environment defensible to a regulator. |

**Why this matters to you commercially:** the components above are the reason a bank can
use this at all. The data never leaves a controlled environment, every action is logged,
every model version is retained, and the compute bill stops when the work stops.

## How to read the rest of this document

From here on, the notebook walks through the seven stages in order. Each stage follows the
same shape:

- **What happens here** — the step in one or two plain sentences.
- **Why it matters to the business** — the commercial or risk consequence.
- Then the actual code, for anyone who wants it. **Skipping every code block loses you nothing.**

Two sections are worth turning to directly if you are short of time:

- **"The cost of being wrong"** (immediately after Step 5) — the money analysis.
- **"Business impact assessment"** (at the end) — risks, recommendations, and how we
  propose to communicate the model's limits to staff, auditors and customers.

---

## Workflow

### Step 1: Gather the tools

**What happens here.** Before any work begins, we load the software libraries the rest of
the process depends on. Nothing is calculated, no data is touched, and no cost is incurred
— this is the equivalent of laying out instruments before a procedure begins.

| Tool being loaded | What it is for |
|---|---|
| `azureml.core` — `Workspace`, `Dataset` | The remote control for our secure Azure environment |
| `pandas` | The spreadsheet engine that holds and manipulates the transaction table |
| `IsolationForest` | The detection algorithm itself — the actual "brain" of this prototype |
| `classification_report` | The scorecard used in Step 5 to mark the model's work |
| `azureml.core.model.Model` | The connection to the model vault used in Step 6 |

**Why it matters to the business.** Each of these is free, open, industry-standard software
used by essentially every financial institution doing this work. There is no proprietary
vendor lock-in at this layer, and no licence cost attached to the detection approach itself.

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Fetch the data, securely

**What happens here.** We connect to the Azure workspace and pull in the registered
transaction dataset — 284,807 card transactions, of which 492 are known frauds. The
data is then loaded into a table structure so it can be worked with, and the first five
rows are displayed purely to confirm the right file arrived.

**Why it matters to the business.** This step is mostly about governance, and it is worth
understanding why we do it this way rather than simply opening a spreadsheet:

- **One governed copy.** Analysts point at a single registered dataset instead of passing
  files around by email. When a regulator asks "which data produced this model?", we can
  answer precisely rather than approximately.
- **Nothing leaves the secure environment.** The data is pulled inside the controlled
  workspace, not onto somebody's laptop.
- **It is reproducible.** The same named dataset will still be there, unchanged and
  versioned, when someone re-runs this in twelve months.
- **It is auditable.** Every access is logged automatically.

**About the data itself — three limitations to keep in mind throughout:**

1. It is a **public research file of European card transactions from September 2013**. It
   is not our data, not our customers, and not our fraud patterns. Any result here is
   indicative only.
2. For privacy reasons the original columns were mathematically disguised before release.
   Twenty-eight of the thirty columns are anonymised and labelled only `V1` to `V28`. **We
   cannot see what they represent.** This becomes a significant issue at Step 7 and in the
   communication plan — we can tell you *that* a transaction was flagged, but not always
   *why*, in business language.
3. Only two columns are meaningful to a human: the transaction `Amount`, and `Time`.

**A practical note.** Running this step downloads roughly 150 MB and takes about four
minutes. You may also see several status and telemetry messages appear beneath the code.
These come from Azure's background logging and are entirely normal — unless the word
`ERROR` or `Traceback` appears, they can be ignored.

*(The first short code cell below simply prints the current folder location. It is a
diagnostic aid used when the notebook is run inside Azure's web console, where the
credentials file has to be uploaded alongside it.)*

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Put every column on the same footing

**What happens here.** Three small but consequential adjustments:

1. The transaction `Amount` column is **rescaled**.
2. The `Time` column is **removed**.
3. The data is split into the **inputs** (what the model is allowed to look at) and the
   **answer key** (whether each transaction was really fraud), which is held back for
   marking in Step 5.

**Why the rescaling matters.** Imagine ranking employees on a combination of their years of
service and their salary. Salary runs into tens of thousands; service runs from 0 to 40. If
you simply add them, salary drowns out service entirely — not because it matters more, but
because its numbers are bigger. The same problem exists here: `Amount` ranges from
€0 to €25,691, while the other columns have already been shrunk to small
numbers. Rescaling puts them on a common footing so the algorithm weighs them on merit
rather than on the size of their units.

**Why removing `Time` matters — and why it is a genuine loss.** In this file, `Time` only
records the number of seconds since the first transaction in the sample. It is a position
in a queue, not a clock: it cannot tell us that a transaction happened at 3 a.m. on a
Sunday. As supplied, it carries no useful signal, so it is dropped.

But be clear about what that means. **Timing is one of the strongest fraud signals that
exists** — time of day, and above all *velocity* (five transactions on one card in ninety
seconds, in three countries). Our prototype is deliberately fighting with one hand tied
behind its back. Restoring proper time-based signals is the single highest-value
improvement on the roadmap at the end of this document, and it is the main reason we expect
a production model to comfortably outperform what you see here.

**Why holding back the answer key matters.** Separating the answers from the inputs is what
makes the Step 5 score trustworthy. The model is marked against information it was never
shown.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Teach the model what "normal" looks like

**What happens here.** This is the step where the model is actually built. It takes about a
minute of computer time and is the technical heart of the prototype.

**The algorithm, without the mathematics.** The method is called an **Isolation Forest**.
The intuition is a game of twenty questions played against a crowded room. To single out a
person of entirely average height, build, and dress, you would need a great many questions.
To single out the one person who is seven feet tall and wearing a gold suit, you need two
or three. The algorithm works exactly like this: it repeatedly splits the data with random
yes/no questions and counts how many questions each transaction needs before it stands
alone. **Transactions that are isolated quickly are the unusual ones**, and those are what
get flagged.

This approach is fast, copes easily with thirty columns, and — crucially — needs no
examples of fraud in order to work. That makes it genuinely useful for spotting *new* fraud
patterns nobody has catalogued yet.

**The one dial you should know about.** The setting `contamination=0.0017` tells the model
to expect about 0.17% of transactions — roughly 476 of our 284,807 — to
be unusual. It is important to understand that this is **an instruction we gave it, not a
discovery it made.** In commercial terms:

> **This dial is the review-queue budget.** Turn it up and you flag more transactions:
> you catch more fraud, you inconvenience more innocent customers, and you need more
> analysts. Turn it down and the opposite happens. It is a business decision about
> risk appetite and staffing, not a technical one — and it belongs to this committee,
> not to the data science team.

**The most important limitation in this entire document.** Look closely at what the model is
given: the inputs only. **It is never shown the 492 known fraud labels.** We are
holding a marked answer key and choosing not to teach from it. That is a legitimate choice
for a first prototype — it tests whether fraud is detectable as pure anomaly — but it means
we are leaving our single most valuable asset unused. Using those labels properly, in what
is called a *supervised* model, is the most important recommendation in this document.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Mark the model's work

**What happens here.** We compare the model's list of flagged transactions against the 492
frauds we already knew about, and print a scorecard.

#### The results

| Group | Hit rate (precision) | Catch rate (recall) | Combined score | Cases in the data |
|---|---|---|---|---|
| Normal transactions | 1.00 | 1.00 | 1.00 | 284,315 |
| **Fraudulent transactions** | **0.29** | **0.28** | **0.28** | **492** |

#### What those numbers mean in customer terms

Numbers like "0.28" are hard to feel. These are the same two results expressed as things
that happen to real people:

- **Catch rate 28%** — *For every 100 genuine frauds committed against our customers,
  we stop 28 and let 72 through.* Across this dataset that is **354 frauds we never even
  looked at.**
- **Hit rate 29%** — *For every 100 customers we flag, roughly 71 have done absolutely
  nothing wrong* and are being inconvenienced — a held payment, a declined card, a phone
  call — because of our system.

#### Two honest caveats about this score

**First, it is probably flattering.** The model was marked on exactly the same data it
learned from — the equivalent of setting a student's exam using the precise questions they
revised. Standard practice is to hold back a portion of the data the model has never seen
and test on that instead. Expect real-world performance to be **somewhat worse than shown
here**, not better.

**Second, the model was scored generously.** It was asked to find *unusual* transactions,
and we are marking it on *fraud*. It is doing the job we gave it; that job is simply not
quite the job we actually need doing.

#### So how bad is this, really?

A 28% catch rate sounds like failure, and as a production system it would be. As a
first prototype, on borrowed data, using no fraud examples and no timing signals, it is a
perfectly respectable starting point — it demonstrates that fraud in this dataset is
partially visible as pure statistical oddity. **The correct reading is not "this does not
work". It is "this works enough to justify building the real thing."** The next section
puts a monetary figure on exactly that.

In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

---

## The cost of being wrong: false alarms versus missed fraud

Every fraud system makes two different kinds of mistake, and they do not cost the same
amount. Getting the balance between them right is the central commercial decision here, so
this section sets out the arithmetic explicitly.

### The two mistakes

| Mistake | What happens | Who feels it |
|---|---|---|
| **False alarm** (false positive) | A legitimate transaction is flagged. An analyst reviews it; the customer may be declined at the till or receive a verification call. | The customer, and the operations budget |
| **Missed fraud** (false negative) | A fraudulent transaction is approved. The loss is absorbed, a chargeback is processed, and the customer's trust is damaged. | The balance sheet, and the customer |

### What each one costs

Two of the figures below are **measured** from the data. The rest are **planning
assumptions** and are labelled as such — they should be replaced with our own finance and
operations numbers before this analysis is used for an actual funding decision.

| Component | Value | Source |
|---|---|---|
| Average value of a fraudulent transaction | €122.21 | **Measured** from the dataset |
| Chargeback handling, investigation, admin | €35.00 | *Assumption* |
| **Total cost of one missed fraud** | **€157.21** | |
| Analyst review time per flagged case | €4.00 | *Assumption* |
| Customer friction / attrition provision | €20.00 | *Assumption* |
| **Total cost of one false alarm** | **€24.00** | |

### The ratio that should drive every tuning decision

> **A missed fraud costs us about 6.6 times as much as a false alarm.**

That single ratio is the most actionable number in this document. It says that, at the
margin, we should be **willing to accept roughly six extra false alarms in order to catch
one more fraud** — and therefore that our current settings, which are tuned conservatively,
are leaving money on the table.

It also marks a genuine change of direction from our previous submission. In CDL10 the
problem we described was a legacy system that flagged *too much*; here the prototype flags
*too little*. Same dial, opposite direction. What matters is that the dial is now being set
by an explicit cost ratio rather than by habit.

### But "catch more fraud" is not automatically the right answer

This is the finding leadership most needs to see, because the intuitive response to a
28% catch rate — "turn it up until we catch most of it" — actively destroys value if
the model's precision is poor. Four scenarios, all costed against this same dataset:

| Scenario | Hit rate | Catch rate | Frauds caught | False alarms | Frauds missed | Total cost |
|---|---|---|---|---|---|---|
| No model (today) | — | 0.00 | 0 | 0 | 492 | **€77,347** |
| This proof of concept | 0.29 | 0.28 | 138 | 338 | 354 | **€63,764** |
| Naive “catch more fraud” retune | 0.10 | 0.80 | 394 | 3,546 | 98 | **€100,511** |
| Production supervised target | 0.50 | 0.80 | 394 | 394 | 98 | **€24,863** |

Read the third row carefully. Simply cranking the sensitivity up until the model catches
80% of fraud — while its hit rate collapses to 10% — produces
3,546 false alarms and a total cost of €100,511. **That is worse than
having no fraud system at all** (€77,347). More alarms is not the same thing as
more protection.

The fourth row is the prize: a model that catches 80% of fraud *while
keeping its hit rate at 50%*. That is a realistic target for a supervised
model built on our own data, and it is why the recommendation is to invest in **model
quality**, not in **alarm volume**.

### Where we stand today

| | Value |
|---|---|
| Cost of doing nothing (all 492 frauds absorbed) | €77,347 |
| Cost with this prototype in place | €63,764 |
| **Net benefit of the prototype** | **€13,583** |
| Share of the available opportunity captured | **18%** |
| Share a production-grade model should capture | **68%** |

Scaled to a portfolio of 10,000,000 transactions a year, the prototype as it stands is
worth roughly **€476,919 a year**; the production target is worth roughly
**€1,842,818 a year**. The gap between those two figures — about
€1,365,899 annually — is the business case for funding the next phase.

*These figures are illustrative and rest on the assumptions tabled above, on a public
dataset that is not our own, and on a model scored against its own training data. They are
intended to size the opportunity and frame the decision, not to be booked.*

### Step 6 (optional): File the model in the vault

**What happens here.** The trained model is saved to a file and formally registered in the
Azure model registry under a name and a version number.

**Why it matters to the business.** This step produces no analysis and no insight, and it is
tempting to regard it as housekeeping. It is not — it is the step that makes everything
else defensible:

- **Provenance.** If a customer disputes a declined transaction eighteen months from now, we
  can identify precisely which version of which model made that decision, and on what data
  it was trained.
- **Rollback.** If a newly deployed model starts behaving badly at 2 a.m., the previous
  version is one command away rather than a rebuild away.
- **Reproducibility.** The model is stored as an artefact, not as somebody's notebook that
  happened to be open on a laptop.
- **Audit and regulation.** Model risk management standards expect a controlled inventory of
  models in use, with versioning and approval history. This registry is that inventory.

In short: Steps 1 to 5 decide whether the model is any good. Step 6 is what allows us to put
it in front of a regulator.

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Seeing the results — how much did we flag?

**What happens here.** A simple bar chart counting the transactions the model called normal
against those it called suspicious.

**How to read it.** You will see one very tall bar (normal) and one almost invisible bar
(flagged). That is the correct and expected shape — fraud is genuinely rare, and a chart in
which those two bars looked similar would be alarming.

**Why it matters to the business.** This is the **staffing chart**. The short bar is
tomorrow's review queue: about 476 cases out of 284,807. It answers the operational
question "how many people would we need?" before we ever get to "how good is it?"

It is also a fast sanity check on the tuning dial from Step 4. Roughly 476 flags against
492 actual frauds means the model is producing about the right *volume* of alerts — it is
the *selection* of which transactions to flag that needs work, not the quantity. That is a
useful distinction: it tells us we have a precision problem, not a capacity problem.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Step 7 (continued): Are we just flagging large transactions?

**What happens here.** A box plot comparing the monetary value of transactions the model
called normal against those it flagged. Each box shows where the middle of the pack sits;
the dots beyond it are unusual values.

**Why we bother.** This chart tests for a specific and very common failure: a model that has
quietly reduced itself to the crude rule "big transaction equals suspicious". If that were
happening, we would not need machine learning at all — and worse, we would be systematically
inconveniencing our highest-value customers while missing everything else.

**And here the data has something genuinely counter-intuitive to say.** Looking at the
492 confirmed frauds in this dataset:

| | Fraudulent transactions | Legitimate transactions |
|---|---|---|
| **Typical (median) value** | **€9.25** | **€22.00** |
| Average value | €122.21 | €88.29 |
| Largest | €2,125.87 | €25,691.16 |

**The typical fraudulent transaction in this data is smaller than the typical legitimate
one — €9.25 against €22.00.** Twenty-seven of the frauds are for
€0 exactly.

That is not a quirk; it is a recognised criminal method. Stolen card numbers are commonly
validated with a string of tiny "card testing" charges that are designed to slip under
exactly the kind of value threshold a traditional rules engine would use. The large average
is the work of a handful of big hits pulling the number upward, which is precisely why the
average is the wrong statistic to manage by.

**Why this matters commercially — this is the argument for the whole programme.** A
conventional rule such as *"review everything over €500"* would miss the majority of the
fraud in this dataset while irritating a great many legitimate high-value customers. The
case for a machine-learning approach is not that it is modern; it is that **the signal we
need is not in the transaction value, and only a model looking at many variables at once
can find it.**

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Step 7 (continued): Why did the model flag what it flagged?

**What happens here.** A technique called **SHAP** is used to open the model up and show
which factors pushed each individual transaction towards being flagged, and how strongly.
The chart is produced for the first 100 transactions to keep it readable.

**How to read it.** Each row is one factor the model considered, with the most influential at
the top. Each dot is a single transaction. Colour shows whether that factor's value was high
(red) or low (blue); position shows whether it pushed the model towards "suspicious"
(right) or "normal" (left).

**Why it matters to the business.** This is the difference between a system we can deploy and
one we cannot. Specifically, it gives us:

- **A defensible answer to a regulator** asking why a particular customer was declined.
- **Something for the contact centre to say** beyond "the computer decided".
- **An early warning of bias** — if the model turned out to lean heavily on a factor
  correlated with geography or demographics, we would need to know before deployment, not
  after a complaint.
- **A guide to improving the model** — the factors at the bottom of the chart contribute
  little and are candidates for removal, which makes the model faster and more stable.

**The honest limitation, and it is a serious one.** As noted at Step 2, the underlying
columns were anonymised for privacy before the data was published. So the best explanation
this prototype can offer is *"the transaction was flagged mainly because of factor V14"* —
which is not an explanation any customer, regulator, or contact-centre agent can use.

This is a limitation of the **public dataset**, not of the technique. When the same method is
applied to our own data, where the columns are things like *transactions in the last hour*,
*distance from the last transaction*, and *merchant category*, SHAP produces genuine
plain-English reason codes. Securing that capability is a prerequisite for deployment, and
it appears in the recommendations below.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

---

# Business impact assessment

The three sections that follow are addressed directly to decision-makers. They cover the
risks of proceeding, what we recommend doing next, and how we propose to talk about this
model's limitations across the organisation.

## Risk assessment and mitigation

| # | Risk | Likelihood | Impact | Mitigation |
|---|---|---|---|---|
| 1 | **Fraud losses continue while the model matures.** At a 28% catch rate, 354 of 492 frauds still get through. | High | High | Run the model **alongside** existing rules and manual review, never instead of them. Treat it as an additional net, not a replacement. |
| 2 | **Customer harm from false declines.** About 7 in 10 flags are innocent customers; a declined card at a checkout is a memorable, sharable experience. | High | High | Use **step-up verification** (a one-time code or app confirmation) rather than a hard decline. Cap the number of interventions per customer per period. Fast, staffed resolution path. |
| 3 | **The model is trained on someone else's data.** European transactions from 2013 — not our customers, not our products, not our fraud patterns. | **Certain** | High | **Do not deploy this model.** Retrain from scratch on our own transaction history. This prototype validates the *approach*, not the model. |
| 4 | **Performance decays as fraud evolves.** Criminals adapt; a model trained on last year's patterns quietly degrades. | High | High | Scheduled retraining, automatic drift monitoring on both input data and catch rate, and a defined threshold that triggers review. |
| 5 | **We cannot explain a decision.** Anonymised inputs mean no plain-English reason codes today. | High *(today)* | High | Build on our own named features so SHAP yields real reason codes. **No deployment before every decline carries a human-readable reason.** |
| 6 | **The score is optimistic.** The model was evaluated against the data it trained on. | **Certain** | Medium | Re-evaluate with a proper held-back test set, and specifically a **time-based** split (train on earlier months, test on later), which is the only honest test for fraud. |
| 7 | **Automation complacency.** Staff start trusting the flags and stop thinking. | Medium | Medium | Keep humans in the decision loop for a defined confidence band, sample and audit both accepted and rejected cases monthly, and report analyst override rates. |
| 8 | **Adversarial adaptation.** Fraudsters probe until they learn where the thresholds sit. | Medium | High | Do not publish thresholds. Vary them. Monitor for probing patterns — clusters of small test transactions are themselves a detectable signal. |
| 9 | **Fairness and conduct exposure.** The model may perform unevenly across customer segments. | Medium | High | Test catch and false-alarm rates by segment before launch and on every retrain; document the results for the conduct committee. |
| 10 | **Cost overrun in cloud compute.** Idle compute bills quietly. | Low | Low | Auto-shutdown on idle compute, budget alerts, and monthly review. |
| 11 | **Data privacy and residency.** Real transaction data is far more sensitive than this public file. | Low | **Severe** | Keep data inside the governed workspace with role-based access; no extracts to local machines; encryption at rest and in transit; DPIA before the pilot begins. |

### The risks of *not* proceeding

For balance, standing still is not a neutral option. On the assumptions above, absorbing all
fraud costs roughly €77,347 across this dataset — about €2,715,780 a
year at 10,000,000 transactions. Rules-only systems also degrade continuously as fraud
patterns shift, and competitors investing in detection will progressively push fraud volume
towards the institutions that have not.

## Recommendations for model improvement and deployment

### Recommendation in one line

> **Fund a pilot, do not deploy this model.** The pipeline is sound and the economics are
> promising; the model itself is a laboratory result trained on borrowed data and should
> never touch a live customer.

### The improvements that matter, in order of value

**1. Use the fraud labels we already have.** *(Highest value by a wide margin.)*
The prototype is *unsupervised* — it was never shown a single example of fraud. We are
sitting on a marked answer key and not teaching from it. Moving to a *supervised* model
(gradient boosting — XGBoost or LightGBM — is the industry standard here) typically lifts
catch rates from the twenties into the seventies and eighties on this class of problem.
This one change accounts for most of the gap between the 18% of value we
capture today and the 68% we are targeting.

**2. Engineer time and velocity features.** Step 3 discarded timing entirely. Restoring it —
transactions per card per hour, time since last transaction, geographic distance from the
previous transaction, deviation from that customer's own established pattern — typically
delivers the largest single improvement after supervision, because velocity is where card
testing and account takeover become obvious.

**3. Tune the threshold against the 6.6:1 cost ratio, not against a fixed setting.**
The `contamination` dial was set to a fixed guess. It should instead be chosen to minimise
expected cost using our real numbers — and revisited whenever those numbers change.

**4. Test honestly, on a time-based split.** Train on earlier months and test on later ones.
Random splits leak future information into the past and flatter the result; fraud is a
moving target and must be tested as one.

**5. Address the imbalance deliberately.** With 0.173% fraud, standard training
methods barely notice the minority class. Class weighting, or careful resampling of the
training set only, is standard practice and materially improves catch rates.

**6. Combine models rather than choosing one.** Keep an anomaly detector like this one
running alongside the supervised model. The supervised model is better at known fraud; the
anomaly detector is the one with a chance of catching a pattern nobody has seen before. They
fail differently, which is exactly why both are worth having.

**7. Enrich the data.** In rough order of expected value: device and browser fingerprint, IP
address and geolocation, merchant category, cardholder tenure and behavioural history,
3-D Secure authentication outcome, and shared-industry fraud consortium signals.

### Deployment path

| Phase | Duration | What happens | Exit criterion |
|---|---|---|---|
| **1. Rebuild on our data** | 6–8 weeks | Supervised model, engineered time/velocity features, honest time-based validation, fairness testing | Catch rate ≥ 80% at hit rate ≥ 50% on held-back data |
| **2. Shadow mode** | 4–6 weeks | Model scores live transactions but **takes no action**. Its decisions are logged and compared against what the current system did. | Predicted performance confirmed on live traffic; no fairness red flags |
| **3. Limited pilot** | 8 weeks | Live on a defined portfolio segment, step-up verification only — no hard declines. Human review retained. | Measured net benefit positive; customer complaint rate within tolerance |
| **4. Champion / challenger** | Ongoing | New models continuously tested against the incumbent on live traffic; the better one is promoted. | — |
| **5. Production with guard-rails** | Ongoing | Auto-decision only in the high-confidence band; a human-review band in the middle; full audit trail and monthly drift reporting. | — |

**A note on the middle band.** We recommend the model never be allowed to make every
decision alone. High-confidence cases can be actioned automatically and clear cases passed
without friction, but a defined middle band should always route to a human analyst. That
pattern — automate the confident, escalate the ambiguous — is what keeps the system both
efficient and defensible.

### What we are asking for

Approval to proceed to **Phase 1** only: a six-to-eight week rebuild on our own transaction
data, returning to this committee with a model validated against held-back data and a
refreshed business case built on Finance's own cost figures rather than the planning
assumptions used here.

## Stakeholder communication plan for model limitations

A fraud model that is oversold is more dangerous than one that underperforms, because the
organisation stops compensating for its weaknesses. This plan is about making sure that
every group hears an accurate version of what this system can and cannot do.

### Four rules we will hold ourselves to

1. **Never quote accuracy.** "99.8% accurate" is technically true, commercially
   meaningless, and actively misleading. It is banned from our reporting.
2. **Always quote catch rate and hit rate together.** Either one alone can be gamed to look
   excellent. Together they tell the truth.
3. **Always state what the model cannot do**, in the same document, at the same prominence
   as what it can.
4. **Maintain a written limitations register**, versioned with the model itself and reviewed
   at every committee meeting.

### Who hears what

| Audience | What they need to hear | Format and cadence |
|---|---|---|
| **Board / Exec committee** | Value at risk, share of opportunity captured, cost of the two error types, the go/no-go decision. Not algorithms. | One-page summary, quarterly and at each phase gate |
| **Fraud operations** | Expected queue volume, what a flag does and does not mean, how to record an override, and that roughly 7 in 10 flags are currently false alarms | Briefing before pilot, then weekly metrics during it |
| **Contact centre** | What to tell a customer whose card was stopped, what we can and cannot explain about why, and the escalation route | Written script and training before any customer-facing action |
| **Compliance / Model risk** | Training data provenance, validation method and its known weaknesses, fairness test results, versioning and audit trail, the explainability gap | Full model documentation pack at each phase gate |
| **Data / IT** | Data residency, access control, retention, retraining schedule, monitoring and alerting thresholds | Technical design review before Phase 2 |
| **Customers** | Never model detail. Clear, non-alarming messaging when a transaction is stopped, a fast resolution path, and no implication of wrongdoing | Embedded in step-up verification messaging |

### How we will describe the model's limits, in plain words

These are the sentences we will actually use, and we will not soften them:

- *"This system finds transactions that are unusual. Most fraud is unusual, but most unusual
  transactions are not fraud. That is why a person reviews the result."*
- *"At present it finds roughly one fraud in four, and about seven of every ten alerts it
  raises turn out to be a legitimate customer. Both of those numbers need to improve
  substantially before it decides anything on its own."*
- *"It was trained on a public research dataset from 2013, not on our customers. Nothing in
  the current results should be read as a prediction of how it will perform here."*
- *"We can currently show which factors drove a decision, but because the research data was
  anonymised we cannot always translate those factors into business language. Closing that
  gap is a condition of deployment, not an enhancement."*

### Escalation triggers

The following are reported immediately rather than waiting for a scheduled review: catch
rate falling more than 5 points below its validated level; false-alarm volume exceeding the
agreed operational ceiling; any measured performance gap between customer segments; any
customer complaint that reaches a regulator; and any evidence of deliberate probing of our
thresholds.

---

## Summary and decision requested

**What we set out to do.** Build a working, end-to-end machine-learning pipeline for card
fraud detection on Azure, and establish whether the approach is worth investing in.

**What we found.**

- The pipeline works end to end — data governance, training, evaluation, model versioning
  and explainability are all in place and repeatable.
- The model catches about **28%** of fraud, and about **71%** of its alerts are
  false alarms. Neither figure is good enough to act on automatically.
- On our stated assumptions it is nonetheless **net positive** — worth roughly
  €13,583 across this dataset, or about €476,919 a year at scale — while
  capturing only **18%** of the value on the table.
- A missed fraud costs about **6.6 times** what a false alarm costs, which tells us to lean
  towards catching more — but *only* by improving the model, since simply raising the alarm
  volume is demonstrably worse than having no model at all.
- The two largest, most fixable weaknesses are that the model **never uses the fraud
  examples we already hold**, and that it **ignores timing and velocity entirely**.

**What we are asking for.** Approval of **Phase 1 only** — a six-to-eight week rebuild on our
own transaction data, using a supervised model with proper time-based validation, returning
to this committee with validated numbers and a business case built on Finance's own costs.

**What we are explicitly not asking for.** Any deployment decision. This model must not make
a decision about a real customer, and nothing in this document should be read as a
recommendation that it does.

---

*Prepared by Uday Kakkar. Python code in this notebook is unmodified from the course
repository; all documentation has been rewritten for a non-technical audience. Figures
quoted are drawn from the Kaggle ULB credit-card fraud dataset and from the model's own
evaluation output, except where explicitly labelled as planning assumptions.*